# TabularAML FeatureGenerator search

This notebook runs automatic feature engineering with `tabularaml.generate.FeatureGenerator` on the poverty prediction training data, then applies the learned feature pipeline to train/test. Outputs are written to `cache/feature_gen/`.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from tabularaml.generate.features import FeatureGenerator
from tabularaml.eval.scorers import Scorer

np.random.seed(42)

DATA_DIR = Path("data")
OUT_DIR = Path("cache/feature_gen")
OUT_DIR.mkdir(parents=True, exist_ok=True)


Added numpy() method to pandas Series


c:\ml_env\lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [2]:
train_features = pd.read_csv(DATA_DIR / "train_hh_features.csv")
train_targets = pd.read_csv(DATA_DIR / "train_hh_gt.csv")

train = train_features.merge(train_targets, on=["survey_id", "hhid"], how="inner")
target_col = "cons_ppp17"
id_cols = ["survey_id", "hhid"]

X = train.drop(columns=id_cols + [target_col])
y = train[target_col]
weights = train["weight"]

print(train.shape, X.shape, y.shape)


(104234, 89) (104234, 86) (104234,)


In [3]:
def weighted_mape(y_true, y_pred, weights):
    y_true = y_true.astype(float)
    y_pred = np.asarray(y_pred, dtype=float)
    w = weights.loc[y_true.index].to_numpy()
    denom = np.clip(np.abs(y_true), 1e-6, None)
    ape = np.abs((y_true - y_pred) / denom)
    return np.average(ape, weights=w)

scorer = Scorer(
    name="weighted_mape",
    scorer=weighted_mape,
    greater_is_better=False,
    extra_params={"weights": weights},
)


In [4]:
mode = "medium"  # "lite", "medium", "best", "extreme"
time_budget_minutes = 30
max_new_feats = 0.75  # fraction of original feature count

fg = FeatureGenerator(
    mode=mode,
    task="regression",
    scorer=scorer,
    max_new_feats=max_new_feats,
    time_budget=time_budget_minutes * 60,
    use_gpu=True,
    log_file="cache/logs/feature_gen_search.log",
)

X_gen, pipeline, generation, interactions = fg.generate(X, y)
print("Original features:", X.shape[1])
print("Engineered features:", X_gen.shape[1])


Starting regression on cuda - 104234 samples, 86 features
Params: gen=25, parents=25, children=150, limit=64, time_budget=1800s.
Gen 0: Train weighted_mape=0.26466, Val weighted_mape=0.30516


KeyboardInterrupt: 

In [ ]:
new_features = sorted(set(X_gen.columns) - set(X.columns))
print("New features:", len(new_features))
new_features[:30]


In [ ]:
test_features = pd.read_csv(DATA_DIR / "test_hh_features.csv")
test_ids = test_features[id_cols].copy()
X_test = test_features.drop(columns=id_cols)

fg.fit(X, y)
X_train_fe = fg.transform(X)
X_test_fe = fg.transform(X_test)

train_out = pd.concat([
    train[id_cols].reset_index(drop=True),
    X_train_fe.reset_index(drop=True),
    y.reset_index(drop=True),
], axis=1)
test_out = pd.concat([
    test_ids.reset_index(drop=True),
    X_test_fe.reset_index(drop=True),
], axis=1)

train_out.to_csv(OUT_DIR / "train_features_fe.csv", index=False)
test_out.to_csv(OUT_DIR / "test_features_fe.csv", index=False)

try:
    fg.save(str(OUT_DIR / "feature_generator.pkl"))
except Exception as exc:
    print(f"Could not save FeatureGenerator: {exc}")
